<a href="https://colab.research.google.com/github/eh3285/audit-home/blob/main/lightgbm/emma/LightGBM_phase0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Home Credit Default Risk Prediction - Emma

## Phase 0 — Reproducible LightGBM audit baseline

**What the run measures:** a seeded 5-fold stratified LightGBM baseline on the
original eight-file aggregated and factorized feature matrix, with a held-out
(out-of-fold) risk score for every labeled applicant.

**What it saves:** CSV file inventory, source/target summaries, feature order and
category-code mappings, fold-level metrics, pooled OOF ROC-AUC / average
precision / Brier / log loss, raw applicant attributes joined by ID, and one
model snapshot per fold. Files go to `audit_artifacts/phase0/` or
`HOME_CREDIT_AUDIT_DIR`.

**Audit scope:** I recreated the original LightGBM feature engineering and model
hyperparameters, with explicit fixed random seed, held-out early stopping and
OOF saving. The training helper is modernized without altering engineered
features.

**Results:** In this notebook, the ROC-AUC of 0.7959, PR-AUC of 0.2917, and Default rate of 8.07% match the original implementations' output, meaning that their result is at least reproducible.

### 0A. Environment, input inventory, and reproducible configuration

Point `HOME_CREDIT_DATA_DIR` to the directory containing the competition CSVs,
or place them in the usual Kaggle / Colab location. Set `HOME_CREDIT_AUDIT_DIR`
to change where results are written.

In [1]:
|from pathlib import Path
from importlib.metadata import version, PackageNotFoundError
import json
import os
import platform
import random
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import PolynomialFeatures, MinMaxScaler
from sklearn.ensemble import RandomForestClassifier

AUDIT_SEED = 42
random.seed(AUDIT_SEED)
np.random.seed(AUDIT_SEED)

REQUIRED_HOME_CREDIT_FILES = [
    "application_train.csv",
    "application_test.csv",
    "previous_application.csv",
    "bureau.csv",
    "bureau_balance.csv",
    "POS_CASH_balance.csv",
    "installments_payments.csv",
    "credit_card_balance.csv",
]
candidate_dirs = [
    os.environ.get("HOME_CREDIT_DATA_DIR"),
    "/kaggle/input/home-credit-default-risk",
    "/content",
    "./data/home-credit-default-risk",
    "./data",
    "../input/home-credit-default-risk",
    ".",
]
candidate_dirs = [Path(p).expanduser() for p in candidate_dirs if p]
complete_dirs = [p for p in candidate_dirs
                 if all((p / name).is_file() for name in REQUIRED_HOME_CREDIT_FILES)]
default_dir = str(complete_dirs[0].resolve()) if complete_dirs else None

AUDIT_OUTPUT_DIR = Path(
    os.environ.get("HOME_CREDIT_AUDIT_DIR", "audit_artifacts/phase0")
).expanduser()
AUDIT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
# Hashing all eight files takes extra time and I/O; enable for exact data checksums.
HASH_RAW_FILES = False
# Preserve the original exploratory Random Forest as an opt-in, not a Phase 0 input.
RUN_LEGACY_RANDOM_FOREST = False
# Optional separate step: refit a full-data model for an unlabeled Kaggle submission.
TRAIN_FULL_SUBMISSION_MODEL = False
# Persist each fold's model to allow precisely reproducible later SHAP experiments.
SAVE_FOLD_MODELS = True

def _package_version(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return "not installed"

environment_manifest = {
    "seed": AUDIT_SEED,
    "python": sys.version.split()[0],
    "platform": platform.platform(),
    "package_versions": {name: _package_version(name) for name in
                         ["numpy", "pandas", "scikit-learn", "lightgbm", "scipy", "shap"]},
    "input_directory": default_dir,
    "expected_input_files": REQUIRED_HOME_CREDIT_FILES,
}
(AUDIT_OUTPUT_DIR / "environment.json").write_text(
    json.dumps(environment_manifest, indent=2), encoding="utf-8"
)
print("Audit output:", AUDIT_OUTPUT_DIR.resolve())
print("Detected data directory:", default_dir or "NOT FOUND")
if default_dir is None:
    print("Set HOME_CREDIT_DATA_DIR to a directory containing all eight CSVs, then rerun.")


Audit output: /content/audit_artifacts/phase0
Detected data directory: /content


### **Step 2: Load the Datasets**
Use `pandas` to load the datasets:

In [2]:
if default_dir is None:
    raise FileNotFoundError("Eight Home Credit CSVs not found. Configure HOME_CREDIT_DATA_DIR in 0A.")

app_train = pd.read_csv(os.path.join(default_dir,'application_train.csv'))
app_test = pd.read_csv(os.path.join(default_dir,'application_test.csv'))

# Display basic information about the dataset
print("Training set shape:", app_train.shape)
print("Testing set shape:", app_test.shape)

app_test.head()


Training set shape: (307511, 122)
Testing set shape: (48744, 121)


,SK_ID_CURR,NAME_CONTRACT_TYPE,CODE_GENDER,FLAG_OWN_CAR,FLAG_OWN_REALTY,CNT_CHILDREN,AMT_INCOME_TOTAL,AMT_CREDIT,AMT_ANNUITY,AMT_GOODS_PRICE,...,FLAG_DOCUMENT_18,FLAG_DOCUMENT_19,FLAG_DOCUMENT_20,FLAG_DOCUMENT_21,AMT_REQ_CREDIT_BUREAU_HOUR,AMT_REQ_CREDIT_BUREAU_DAY,AMT_REQ_CREDIT_BUREAU_WEEK,AMT_REQ_CREDIT_BUREAU_MON,AMT_REQ_CREDIT_BUREAU_QRT,AMT_REQ_CREDIT_BUREAU_YEAR
0,100001,Cash loans,F,N,Y,0,135000.0,568800.0,20560.5,450000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,0.0
1,100005,Cash loans,M,N,Y,0,99000.0,222768.0,17370.0,180000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
2,100013,Cash loans,M,Y,Y,0,202500.0,663264.0,69777.0,630000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,1.0,4.0
3,100028,Cash loans,F,N,Y,2,315000.0,1575000.0,49018.5,1575000.0,...,0,0,0,0,0.0,0.0,0.0,0.0,0.0,3.0
4,100038,Cash loans,M,Y,N,1,180000.0,625500.0,32067.0,625500.0,...,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN


### 0B. Verify the raw applicant population and preserve audit attributes

The competition's labeled training set provides the audit population. The
unlabeled test set cannot be used to calculate accuracy or fairness metrics.
The saved `audit_reference_raw` table is keyed by `SK_ID_CURR`; none of these
columns is substituted for the actual engineered LightGBM feature table.


In [3]:
import hashlib

if default_dir is None:
    raise FileNotFoundError(
        "Missing Home Credit CSVs. Set HOME_CREDIT_DATA_DIR to the folder "
        "containing all eight original competition CSVs, then rerun 0A."
    )

def _file_sha256(path):
    digest = hashlib.sha256()
    with open(path, "rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

raw_file_inventory = pd.DataFrame([
    {
        "filename": name,
        "size_bytes": (Path(default_dir) / name).stat().st_size,
        "sha256": _file_sha256(Path(default_dir) / name) if HASH_RAW_FILES else None,
    }
    for name in REQUIRED_HOME_CREDIT_FILES
])
raw_file_inventory.to_csv(AUDIT_OUTPUT_DIR / "raw_file_inventory.csv", index=False)

assert "TARGET" in app_train.columns and "TARGET" not in app_test.columns
assert app_train["SK_ID_CURR"].notna().all() and app_test["SK_ID_CURR"].notna().all()
assert app_train["SK_ID_CURR"].is_unique and app_test["SK_ID_CURR"].is_unique
assert not set(app_train["SK_ID_CURR"]).intersection(app_test["SK_ID_CURR"]), (
    "Labeled and competition-test IDs overlap."
)
assert app_train["TARGET"].notna().all()
assert set(app_train["TARGET"].unique()).issubset({0, 1})

# Freeze raw attributes BEFORE the optional exploratory cells encode or modify them.
AUDIT_RAW_ATTRIBUTES = [
    "CODE_GENDER", "DAYS_BIRTH", "NAME_EDUCATION_TYPE",
    "NAME_FAMILY_STATUS", "NAME_INCOME_TYPE", "AMT_INCOME_TOTAL",
    "AMT_CREDIT", "AMT_ANNUITY", "DAYS_EMPLOYED", "EXT_SOURCE_1",
    "EXT_SOURCE_2", "EXT_SOURCE_3", "FLAG_OWN_CAR",
    "FLAG_OWN_REALTY", "REGION_RATING_CLIENT",
]
audit_reference_raw = app_train[
    ["SK_ID_CURR"] + [col for col in AUDIT_RAW_ATTRIBUTES if col in app_train]
].copy()

raw_summary = {
    "train_rows": int(len(app_train)),
    "competition_test_rows": int(len(app_test)),
    "train_columns_including_target": int(app_train.shape[1]),
    "test_columns": int(app_test.shape[1]),
    "label_1_count": int(app_train["TARGET"].sum()),
    "label_1_rate": float(app_train["TARGET"].mean()),
    "unique_train_ids": int(app_train["SK_ID_CURR"].nunique()),
    "unique_test_ids": int(app_test["SK_ID_CURR"].nunique()),
    "days_employed_365243_train": int((app_train["DAYS_EMPLOYED"] == 365243).sum()),
    "days_employed_365243_test": int((app_test["DAYS_EMPLOYED"] == 365243).sum()),
    "protected_or_proxy_attributes_preserved": list(audit_reference_raw.columns[1:]),
}
(AUDIT_OUTPUT_DIR / "raw_population_summary.json").write_text(
    json.dumps(raw_summary, indent=2), encoding="utf-8"
)
print("Labeled applicants:", f'{raw_summary["train_rows"]:,}')
print("Competition-test applicants:", f'{raw_summary["competition_test_rows"]:,}')
print("TARGET=1 prevalence:", f'{raw_summary["label_1_rate"]:.3%}')
display(raw_file_inventory[["filename", "size_bytes"]])
display(app_train.isna().mean().sort_values(ascending=False).head(12).rename("missing_fraction").to_frame())


Labeled applicants: 307,511
Competition-test applicants: 48,744
TARGET=1 prevalence: 8.073%


,filename,size_bytes
0,application_train.csv,166133370
1,application_test.csv,26567651
2,previous_application.csv,404973293
3,bureau.csv,170016717
4,bureau_balance.csv,375592889
5,POS_CASH_balance.csv,392703158
6,installments_payments.csv,723118349
7,credit_card_balance.csv,424582605


,missing_fraction
COMMONAREA_AVG,0.698723
COMMONAREA_MODE,0.698723
COMMONAREA_MEDI,0.698723
NONLIVINGAPARTMENTS_MEDI,0.694330
NONLIVINGAPARTMENTS_MODE,0.694330
NONLIVINGAPARTMENTS_AVG,0.694330
FONDKAPREMONT_MODE,0.683862
LIVINGAPARTMENTS_AVG,0.683550
LIVINGAPARTMENTS_MEDI,0.683550
LIVINGAPARTMENTS_MODE,0.683550


## Original eight-file feature engineering (unchanged)

These functions reproduce the preprocessing consumed by the final LightGBM.
The audit's extra metadata collection does **not** change feature values.


In [4]:
def get_balance_data():
    pos_dtype = {
        'SK_ID_PREV':np.uint32, 'SK_ID_CURR':np.uint32, 'MONTHS_BALANCE':np.int32, 'SK_DPD':np.int32,
        'SK_DPD_DEF':np.int32, 'CNT_INSTALMENT':np.float32,'CNT_INSTALMENT_FUTURE':np.float32
    }

    install_dtype = {
        'SK_ID_PREV':np.uint32, 'SK_ID_CURR':np.uint32, 'NUM_INSTALMENT_NUMBER':np.int32, 'NUM_INSTALMENT_VERSION':np.float32,
        'DAYS_INSTALMENT':np.float32, 'DAYS_ENTRY_PAYMENT':np.float32, 'AMT_INSTALMENT':np.float32, 'AMT_PAYMENT':np.float32
    }

    card_dtype = {
        'SK_ID_PREV':np.uint32, 'SK_ID_CURR':np.uint32, 'MONTHS_BALANCE':np.int16,
        'AMT_CREDIT_LIMIT_ACTUAL':np.int32, 'CNT_DRAWINGS_CURRENT':np.int32, 'SK_DPD':np.int32,'SK_DPD_DEF':np.int32,
        'AMT_BALANCE':np.float32, 'AMT_DRAWINGS_ATM_CURRENT':np.float32, 'AMT_DRAWINGS_CURRENT':np.float32,
        'AMT_DRAWINGS_OTHER_CURRENT':np.float32, 'AMT_DRAWINGS_POS_CURRENT':np.float32, 'AMT_INST_MIN_REGULARITY':np.float32,
        'AMT_PAYMENT_CURRENT':np.float32, 'AMT_PAYMENT_TOTAL_CURRENT':np.float32, 'AMT_RECEIVABLE_PRINCIPAL':np.float32,
        'AMT_RECIVABLE':np.float32, 'AMT_TOTAL_RECEIVABLE':np.float32, 'CNT_DRAWINGS_ATM_CURRENT':np.float32,
        'CNT_DRAWINGS_OTHER_CURRENT':np.float32, 'CNT_DRAWINGS_POS_CURRENT':np.float32, 'CNT_INSTALMENT_MATURE_CUM':np.float32
    }

    pos_bal = pd.read_csv(os.path.join(default_dir,'POS_CASH_balance.csv'), dtype=pos_dtype)
    install = pd.read_csv(os.path.join(default_dir,'installments_payments.csv'), dtype=install_dtype)
    card_bal = pd.read_csv(os.path.join(default_dir, 'credit_card_balance.csv'), dtype=card_dtype)

    return pos_bal, install, card_bal

# Loaded once by get_dataset() in the final LightGBM pipeline.

In [5]:
def get_apps_processed(apps):
    """
    feature engineering for apps
    """

    # 1.EXT_SOURCE
    apps['APPS_EXT_SOURCE_MEAN'] = apps[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].mean(axis = 1)
    apps['APPS_EXT_SOURCE_STD'] = apps[['EXT_SOURCE_1', 'EXT_SOURCE_2', 'EXT_SOURCE_3']].std(axis=1)
    apps['APPS_EXT_SOURCE_STD'] = apps['APPS_EXT_SOURCE_STD'].fillna(apps['APPS_EXT_SOURCE_STD'].mean())

    # AMT_CREDIT
    apps['APPS_ANNUITY_CREDIT_RATIO'] = apps['AMT_ANNUITY']/apps['AMT_CREDIT']
    apps['APPS_GOODS_CREDIT_RATIO'] = apps['AMT_GOODS_PRICE']/apps['AMT_CREDIT']

    # AMT_INCOME_TOTAL
    apps['APPS_ANNUITY_INCOME_RATIO'] = apps['AMT_ANNUITY']/apps['AMT_INCOME_TOTAL']
    apps['APPS_CREDIT_INCOME_RATIO'] = apps['AMT_CREDIT']/apps['AMT_INCOME_TOTAL']
    apps['APPS_GOODS_INCOME_RATIO'] = apps['AMT_GOODS_PRICE']/apps['AMT_INCOME_TOTAL']
    apps['APPS_CNT_FAM_INCOME_RATIO'] = apps['AMT_INCOME_TOTAL']/apps['CNT_FAM_MEMBERS']

    # DAYS_BIRTH, DAYS_EMPLOYED
    apps['APPS_EMPLOYED_BIRTH_RATIO'] = apps['DAYS_EMPLOYED']/apps['DAYS_BIRTH']
    apps['APPS_INCOME_EMPLOYED_RATIO'] = apps['AMT_INCOME_TOTAL']/apps['DAYS_EMPLOYED']
    apps['APPS_INCOME_BIRTH_RATIO'] = apps['AMT_INCOME_TOTAL']/apps['DAYS_BIRTH']
    apps['APPS_CAR_BIRTH_RATIO'] = apps['OWN_CAR_AGE'] / apps['DAYS_BIRTH']
    apps['APPS_CAR_EMPLOYED_RATIO'] = apps['OWN_CAR_AGE'] / apps['DAYS_EMPLOYED']

    return apps

In [6]:
def get_prev_processed(prev):
    """
    feature engineering
    for previous application credit history
    """
    prev['PREV_CREDIT_DIFF'] = prev['AMT_APPLICATION'] - prev['AMT_CREDIT']
    prev['PREV_GOODS_DIFF'] = prev['AMT_APPLICATION'] - prev['AMT_GOODS_PRICE']
    prev['PREV_CREDIT_APPL_RATIO'] = prev['AMT_CREDIT']/prev['AMT_APPLICATION']
    prev['PREV_GOODS_APPL_RATIO'] = prev['AMT_GOODS_PRICE']/prev['AMT_APPLICATION']

    prev['DAYS_FIRST_DRAWING'] = prev['DAYS_FIRST_DRAWING'].replace(365243, np.nan)
    prev['DAYS_FIRST_DUE'] = prev['DAYS_FIRST_DUE'].replace(365243, np.nan)
    prev['DAYS_LAST_DUE_1ST_VERSION'] = prev['DAYS_LAST_DUE_1ST_VERSION'].replace(365243, np.nan)
    prev['DAYS_LAST_DUE'] = prev['DAYS_LAST_DUE'].replace(365243, np.nan)
    prev['DAYS_TERMINATION'] = prev['DAYS_TERMINATION'].replace(365243, np.nan)

    prev['PREV_DAYS_LAST_DUE_DIFF'] = prev['DAYS_LAST_DUE_1ST_VERSION'] - prev['DAYS_LAST_DUE']

    all_pay = prev['AMT_ANNUITY'] * prev['CNT_PAYMENT']
    prev['PREV_INTERESTS_RATE'] = (all_pay/prev['AMT_CREDIT'] - 1)/prev['CNT_PAYMENT']

    return prev

def get_prev_amt_agg(prev):
    """
    feature enginnering for the previous credit application
    """
    agg_dict = {
      'SK_ID_CURR':['count'],
      'AMT_CREDIT':['mean', 'max', 'sum'],
      'AMT_ANNUITY':['mean', 'max', 'sum'],
      'AMT_APPLICATION':['mean', 'max', 'sum'],
      'AMT_DOWN_PAYMENT':['mean', 'max', 'sum'],
      'AMT_GOODS_PRICE':['mean', 'max', 'sum'],
      'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
      'DAYS_DECISION': ['min', 'max', 'mean'],
      'CNT_PAYMENT': ['mean', 'sum'],

      'PREV_CREDIT_DIFF':['mean', 'max', 'sum'],
      'PREV_CREDIT_APPL_RATIO':['mean', 'max'],
      'PREV_GOODS_DIFF':['mean', 'max', 'sum'],
      'PREV_GOODS_APPL_RATIO':['mean', 'max'],
      'PREV_DAYS_LAST_DUE_DIFF':['mean', 'max', 'sum'],
      'PREV_INTERESTS_RATE':['mean', 'max']
    }

    prev_group = prev.groupby('SK_ID_CURR')
    prev_amt_agg = prev_group.agg(agg_dict)
    prev_amt_agg.columns = ["PREV_"+ "_".join(x).upper() for x in prev_amt_agg.columns.ravel()]

    return prev_amt_agg

def get_prev_refused_appr_agg(prev):
    """
    PREV_APPROVED_COUNT : Credit application approved count
    PREV_REFUSED_COUNT :  Credit application refused count
    """
    prev_refused_appr_group = prev[prev['NAME_CONTRACT_STATUS'].isin(['Approved', 'Refused'])].groupby([ 'SK_ID_CURR', 'NAME_CONTRACT_STATUS'])
    prev_refused_appr_agg = prev_refused_appr_group['SK_ID_CURR'].count().unstack()
    prev_refused_appr_agg.columns = ['PREV_APPROVED_COUNT', 'PREV_REFUSED_COUNT' ]
    prev_refused_appr_agg = prev_refused_appr_agg.fillna(0)
    return prev_refused_appr_agg


def get_prev_days365_agg(prev):
    """
    DAYS_DESCISION means How many days have been take since the previous credit application made.
    Somehow this feature is important.
    """
    cond_days365 = prev['DAYS_DECISION'] > -365
    prev_days365_group = prev[cond_days365].groupby('SK_ID_CURR')
    agg_dict = {
      'SK_ID_CURR':['count'],
      'AMT_CREDIT':['mean', 'max', 'sum'],
      'AMT_ANNUITY':['mean', 'max', 'sum'],
      'AMT_APPLICATION':['mean', 'max', 'sum'],
      'AMT_DOWN_PAYMENT':['mean', 'max', 'sum'],
      'AMT_GOODS_PRICE':['mean', 'max', 'sum'],
      'RATE_DOWN_PAYMENT': ['min', 'max', 'mean'],
      'DAYS_DECISION': ['min', 'max', 'mean'],
      'CNT_PAYMENT': ['mean', 'sum'],

      'PREV_CREDIT_DIFF':['mean', 'max', 'sum'],
      'PREV_CREDIT_APPL_RATIO':['mean', 'max'],
      'PREV_GOODS_DIFF':['mean', 'max', 'sum'],
      'PREV_GOODS_APPL_RATIO':['mean', 'max'],
      'PREV_DAYS_LAST_DUE_DIFF':['mean', 'max', 'sum'],
      'PREV_INTERESTS_RATE':['mean', 'max']
    }

    prev_days365_agg = prev_days365_group.agg(agg_dict)

    # multi index
    prev_days365_agg.columns = ["PREV_D365_"+ "_".join(x).upper() for x in prev_days365_agg.columns.ravel()]

    return prev_days365_agg

def get_prev_agg(prev):
    prev = get_prev_processed(prev)
    prev_amt_agg = get_prev_amt_agg(prev)
    prev_refused_appr_agg = get_prev_refused_appr_agg(prev)
    prev_days365_agg = get_prev_days365_agg(prev)

    # prev_amt_agg
    prev_agg = prev_amt_agg.merge(prev_refused_appr_agg, on='SK_ID_CURR', how='left')
    prev_agg = prev_agg.merge(prev_days365_agg, on='SK_ID_CURR', how='left')
    # SK_ID_CURR APPROVED_COUNT REFUSED_COUNT
    prev_agg['PREV_REFUSED_RATIO'] = prev_agg['PREV_REFUSED_COUNT']/prev_agg['PREV_SK_ID_CURR_COUNT']
    prev_agg['PREV_APPROVED_RATIO'] = prev_agg['PREV_APPROVED_COUNT']/prev_agg['PREV_SK_ID_CURR_COUNT']
    # 'PREV_REFUSED_COUNT', 'PREV_APPROVED_COUNT' drop
    prev_agg = prev_agg.drop(['PREV_REFUSED_COUNT', 'PREV_APPROVED_COUNT'], axis=1)

    return prev_agg

In [7]:
def get_bureau_processed(bureau):
    bureau['BUREAU_ENDDATE_FACT_DIFF'] = bureau['DAYS_CREDIT_ENDDATE'] - bureau['DAYS_ENDDATE_FACT']
    bureau['BUREAU_CREDIT_FACT_DIFF'] = bureau['DAYS_CREDIT'] - bureau['DAYS_ENDDATE_FACT']
    bureau['BUREAU_CREDIT_ENDDATE_DIFF'] = bureau['DAYS_CREDIT'] - bureau['DAYS_CREDIT_ENDDATE']
    bureau['BUREAU_CREDIT_DEBT_RATIO']=bureau['AMT_CREDIT_SUM_DEBT']/bureau['AMT_CREDIT_SUM']
    bureau['BUREAU_CREDIT_DEBT_DIFF'] = bureau['AMT_CREDIT_SUM_DEBT'] - bureau['AMT_CREDIT_SUM']

    bureau['BUREAU_IS_DPD'] = bureau['CREDIT_DAY_OVERDUE'].apply(lambda x: 1 if x > 0 else 0)
    bureau['BUREAU_IS_DPD_OVER120'] = bureau['CREDIT_DAY_OVERDUE'].apply(lambda x: 1 if x >120 else 0)

    return bureau

def get_bureau_day_amt_agg(bureau):
    bureau_agg_dict = {
    'SK_ID_BUREAU':['count'],
    'DAYS_CREDIT':['min', 'max', 'mean'],
    'CREDIT_DAY_OVERDUE':['min', 'max', 'mean'],
    'DAYS_CREDIT_ENDDATE':['min', 'max', 'mean'],
    'DAYS_ENDDATE_FACT':['min', 'max', 'mean'],
    'AMT_CREDIT_MAX_OVERDUE': ['max', 'mean'],
    'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
    'AMT_CREDIT_SUM_OVERDUE': ['max', 'mean', 'sum'],
    'AMT_ANNUITY': ['max', 'mean', 'sum'],

    'BUREAU_ENDDATE_FACT_DIFF':['min', 'max', 'mean'],
    'BUREAU_CREDIT_FACT_DIFF':['min', 'max', 'mean'],
    'BUREAU_CREDIT_ENDDATE_DIFF':['min', 'max', 'mean'],
    'BUREAU_CREDIT_DEBT_RATIO':['min', 'max', 'mean'],
    'BUREAU_CREDIT_DEBT_DIFF':['min', 'max', 'mean'],
    'BUREAU_IS_DPD':['mean', 'sum'],
    'BUREAU_IS_DPD_OVER120':['mean', 'sum']
    }

    bureau_grp = bureau.groupby('SK_ID_CURR')
    bureau_day_amt_agg = bureau_grp.agg(bureau_agg_dict)
    bureau_day_amt_agg.columns = ['BUREAU_'+('_').join(column).upper() for column in bureau_day_amt_agg.columns.ravel()]
    # SK_ID_CURR reset_index()
    bureau_day_amt_agg = bureau_day_amt_agg.reset_index()
    #print('bureau_day_amt_agg shape:', bureau_day_amt_agg.shape)
    return bureau_day_amt_agg

def get_bureau_active_agg(bureau):
    cond_active = bureau['CREDIT_ACTIVE'] == 'Active'
    bureau_active_grp = bureau[cond_active].groupby(['SK_ID_CURR'])
    bureau_agg_dict = {
      'SK_ID_BUREAU':['count'],
      'DAYS_CREDIT':['min', 'max', 'mean'],
      'CREDIT_DAY_OVERDUE':['min', 'max', 'mean'],
      'DAYS_CREDIT_ENDDATE':['min', 'max', 'mean'],
      'DAYS_ENDDATE_FACT':['min', 'max', 'mean'],
      'AMT_CREDIT_MAX_OVERDUE': ['max', 'mean'],
      'AMT_CREDIT_SUM': ['max', 'mean', 'sum'],
      'AMT_CREDIT_SUM_DEBT': ['max', 'mean', 'sum'],
      'AMT_CREDIT_SUM_OVERDUE': ['max', 'mean', 'sum'],
      'AMT_ANNUITY': ['max', 'mean', 'sum'],

      'BUREAU_ENDDATE_FACT_DIFF':['min', 'max', 'mean'],
      'BUREAU_CREDIT_FACT_DIFF':['min', 'max', 'mean'],
      'BUREAU_CREDIT_ENDDATE_DIFF':['min', 'max', 'mean'],
      'BUREAU_CREDIT_DEBT_RATIO':['min', 'max', 'mean'],
      'BUREAU_CREDIT_DEBT_DIFF':['min', 'max', 'mean'],
      'BUREAU_IS_DPD':['mean', 'sum'],
      'BUREAU_IS_DPD_OVER120':['mean', 'sum']
      }

    bureau_active_agg = bureau_active_grp.agg(bureau_agg_dict)
    bureau_active_agg.columns = ['BUREAU_ACT_'+('_').join(column).upper() for column in bureau_active_agg.columns.ravel()]
    bureau_active_agg = bureau_active_agg.reset_index()

    return bureau_active_agg


def get_bureau_bal_agg(bureau, bureau_bal):
    bureau_bal = bureau_bal.merge(bureau[['SK_ID_CURR', 'SK_ID_BUREAU']], on='SK_ID_BUREAU', how='left')
    bureau_bal['BUREAU_BAL_IS_DPD'] = bureau_bal['STATUS'].apply(lambda x: 1 if x in['1','2','3','4','5']  else 0)
    bureau_bal['BUREAU_BAL_IS_DPD_OVER120'] = bureau_bal['STATUS'].apply(lambda x: 1 if x =='5'  else 0)
    bureau_bal_grp = bureau_bal.groupby('SK_ID_CURR')

    bureau_bal_agg_dict = {
        'SK_ID_CURR':['count'],
        'MONTHS_BALANCE':['min', 'max', 'mean'],
        'BUREAU_BAL_IS_DPD':['mean', 'sum'],
        'BUREAU_BAL_IS_DPD_OVER120':['mean', 'sum']
    }

    bureau_bal_agg = bureau_bal_grp.agg(bureau_bal_agg_dict)
    bureau_bal_agg.columns = [ 'BUREAU_BAL_'+('_').join(column).upper() for column in bureau_bal_agg.columns.ravel() ]

    bureau_bal_agg = bureau_bal_agg.reset_index()
    return bureau_bal_agg

def get_bureau_agg(bureau, bureau_bal):
    bureau = get_bureau_processed(bureau)
    bureau_day_amt_agg = get_bureau_day_amt_agg(bureau)
    bureau_active_agg = get_bureau_active_agg(bureau)
    bureau_bal_agg = get_bureau_bal_agg(bureau, bureau_bal)

    bureau_agg = bureau_day_amt_agg.merge(bureau_active_agg, on='SK_ID_CURR', how='left')
    bureau_agg['BUREAU_ACT_IS_DPD_RATIO'] = bureau_agg['BUREAU_ACT_BUREAU_IS_DPD_SUM']/bureau_agg['BUREAU_SK_ID_BUREAU_COUNT']
    bureau_agg['BUREAU_ACT_IS_DPD_OVER120_RATIO'] = bureau_agg['BUREAU_ACT_BUREAU_IS_DPD_OVER120_SUM']/bureau_agg['BUREAU_SK_ID_BUREAU_COUNT']

    bureau_agg = bureau_agg.merge(bureau_bal_agg, on='SK_ID_CURR', how='left')
    #bureau_agg = bureau_agg.merge(bureau_days750_agg, on='SK_ID_CURR', how='left')

    return bureau_agg

In [8]:
def get_pos_bal_agg(pos_bal):
    cond_over_0 = pos_bal['SK_DPD'] > 0
    cond_100 = (pos_bal['SK_DPD'] < 100) & (pos_bal['SK_DPD'] > 0)
    cond_over_100 = (pos_bal['SK_DPD'] >= 100)

    pos_bal['POS_IS_DPD'] = pos_bal['SK_DPD'].apply(lambda x: 1 if x > 0 else 0)
    pos_bal['POS_IS_DPD_UNDER_120'] = pos_bal['SK_DPD'].apply(lambda x:1 if (x > 0) & (x <120) else 0 )
    pos_bal['POS_IS_DPD_OVER_120'] = pos_bal['SK_DPD'].apply(lambda x:1 if x >= 120 else 0)

    pos_bal_grp = pos_bal.groupby('SK_ID_CURR')
    pos_bal_agg_dict = {
        'SK_ID_CURR':['count'],
        'MONTHS_BALANCE':['min', 'mean', 'max'],
        'SK_DPD':['min', 'max', 'mean', 'sum'],
        'CNT_INSTALMENT':['min', 'max', 'mean', 'sum'],
        'CNT_INSTALMENT_FUTURE':['min', 'max', 'mean', 'sum'],

        'POS_IS_DPD':['mean', 'sum'],
        'POS_IS_DPD_UNDER_120':['mean', 'sum'],
        'POS_IS_DPD_OVER_120':['mean', 'sum']
    }

    pos_bal_agg = pos_bal_grp.agg(pos_bal_agg_dict)

    pos_bal_agg.columns = [('POS_')+('_').join(column).upper() for column in pos_bal_agg.columns.ravel()]

    cond_months = pos_bal['MONTHS_BALANCE'] > -20
    pos_bal_m20_grp = pos_bal[cond_months].groupby('SK_ID_CURR')
    pos_bal_m20_agg_dict = {
        'SK_ID_CURR':['count'],
        'MONTHS_BALANCE':['min', 'mean', 'max'],
        'SK_DPD':['min', 'max', 'mean', 'sum'],
        'CNT_INSTALMENT':['min', 'max', 'mean', 'sum'],
        'CNT_INSTALMENT_FUTURE':['min', 'max', 'mean', 'sum'],

        'POS_IS_DPD':['mean', 'sum'],
        'POS_IS_DPD_UNDER_120':['mean', 'sum'],
        'POS_IS_DPD_OVER_120':['mean', 'sum']
    }

    pos_bal_m20_agg = pos_bal_m20_grp.agg(pos_bal_m20_agg_dict)

    pos_bal_m20_agg.columns = [('POS_M20')+('_').join(column).upper() for column in pos_bal_m20_agg.columns.ravel()]
    pos_bal_agg = pos_bal_agg.merge(pos_bal_m20_agg, on='SK_ID_CURR', how='left')
    pos_bal_agg = pos_bal_agg.reset_index()


    return pos_bal_agg

In [9]:
def get_install_agg(install):
    install['AMT_DIFF'] = install['AMT_INSTALMENT'] - install['AMT_PAYMENT']
    install['AMT_RATIO'] =  (install['AMT_PAYMENT'] +1)/ (install['AMT_INSTALMENT'] + 1)
    install['SK_DPD'] = install['DAYS_ENTRY_PAYMENT'] - install['DAYS_INSTALMENT']

    install['INS_IS_DPD'] = install['SK_DPD'].apply(lambda x: 1 if x > 0 else 0)
    install['INS_IS_DPD_UNDER_120'] = install['SK_DPD'].apply(lambda x:1 if (x > 0) & (x <120) else 0 )
    install['INS_IS_DPD_OVER_120'] = install['SK_DPD'].apply(lambda x:1 if x >= 120 else 0)

    install_grp = install.groupby('SK_ID_CURR')

    install_agg_dict = {
        'SK_ID_CURR':['count'],
        'NUM_INSTALMENT_VERSION':['nunique'],
        'DAYS_ENTRY_PAYMENT':['mean', 'max', 'sum'],
        'DAYS_INSTALMENT':['mean', 'max', 'sum'],
        'AMT_INSTALMENT':['mean', 'max', 'sum'],
        'AMT_PAYMENT':['mean', 'max','sum'],

        'AMT_DIFF':['mean','min', 'max','sum'],
        'AMT_RATIO':['mean', 'max'],
        'SK_DPD':['mean', 'min', 'max'],
        'INS_IS_DPD':['mean', 'sum'],
        'INS_IS_DPD_UNDER_120':['mean', 'sum'],
        'INS_IS_DPD_OVER_120':['mean', 'sum']
    }

    install_agg = install_grp.agg(install_agg_dict)
    install_agg.columns = ['INS_'+('_').join(column).upper() for column in install_agg.columns.ravel()]

    cond_day = install['DAYS_ENTRY_PAYMENT'] >= -365
    install_d365_grp = install[cond_day].groupby('SK_ID_CURR')
    install_d365_agg_dict = {
        'SK_ID_CURR':['count'],
        'NUM_INSTALMENT_VERSION':['nunique'],
        'DAYS_ENTRY_PAYMENT':['mean', 'max', 'sum'],
        'DAYS_INSTALMENT':['mean', 'max', 'sum'],
        'AMT_INSTALMENT':['mean', 'max', 'sum'],
        'AMT_PAYMENT':['mean', 'max','sum'],

        'AMT_DIFF':['mean','min', 'max','sum'],
        'AMT_RATIO':['mean', 'max'],
        'SK_DPD':['mean', 'min', 'max'],
        'INS_IS_DPD':['mean', 'sum'],
        'INS_IS_DPD_UNDER_120':['mean', 'sum'],
        'INS_IS_DPD_OVER_120':['mean', 'sum']
    }

    install_d365_agg = install_d365_grp.agg(install_d365_agg_dict)
    install_d365_agg.columns = ['INS_D365'+('_').join(column).upper() for column in install_d365_agg.columns.ravel()]

    install_agg = install_agg.merge(install_d365_agg, on='SK_ID_CURR', how='left')
    install_agg = install_agg.reset_index()

    return install_agg

In [10]:
def get_card_bal_agg(card_bal):
    card_bal['BALANCE_LIMIT_RATIO'] = card_bal['AMT_BALANCE']/card_bal['AMT_CREDIT_LIMIT_ACTUAL']
    card_bal['DRAWING_LIMIT_RATIO'] = card_bal['AMT_DRAWINGS_CURRENT'] / card_bal['AMT_CREDIT_LIMIT_ACTUAL']

    card_bal['CARD_IS_DPD'] = card_bal['SK_DPD'].apply(lambda x: 1 if x > 0 else 0)
    card_bal['CARD_IS_DPD_UNDER_120'] = card_bal['SK_DPD'].apply(lambda x:1 if (x > 0) & (x <120) else 0 )
    card_bal['CARD_IS_DPD_OVER_120'] = card_bal['SK_DPD'].apply(lambda x:1 if x >= 120 else 0)

    card_bal_grp = card_bal.groupby('SK_ID_CURR')
    card_bal_agg_dict = {
        'SK_ID_CURR':['count'],
        'AMT_BALANCE':['max'],
        'AMT_CREDIT_LIMIT_ACTUAL':['max'],
        'AMT_DRAWINGS_ATM_CURRENT': ['max', 'sum'],
        'AMT_DRAWINGS_CURRENT': ['max', 'sum'],
        'AMT_DRAWINGS_POS_CURRENT': ['max', 'sum'],
        'AMT_INST_MIN_REGULARITY': ['max', 'mean'],
        'AMT_PAYMENT_TOTAL_CURRENT': ['max','sum'],
        'AMT_TOTAL_RECEIVABLE': ['max', 'mean'],
        'CNT_DRAWINGS_ATM_CURRENT': ['max','sum'],
        'CNT_DRAWINGS_CURRENT': ['max', 'mean', 'sum'],
        'CNT_DRAWINGS_POS_CURRENT': ['mean'],
        'SK_DPD': ['mean', 'max', 'sum'],

        'BALANCE_LIMIT_RATIO':['min','max'],
        'DRAWING_LIMIT_RATIO':['min', 'max'],
        'CARD_IS_DPD':['mean', 'sum'],
        'CARD_IS_DPD_UNDER_120':['mean', 'sum'],
        'CARD_IS_DPD_OVER_120':['mean', 'sum']
    }
    card_bal_agg = card_bal_grp.agg(card_bal_agg_dict)
    card_bal_agg.columns = ['CARD_'+('_').join(column).upper() for column in card_bal_agg.columns.ravel()]

    card_bal_agg = card_bal_agg.reset_index()

    cond_month = card_bal.MONTHS_BALANCE >= -3
    card_bal_m3_grp = card_bal[cond_month].groupby('SK_ID_CURR')
    card_bal_m3_agg = card_bal_m3_grp.agg(card_bal_agg_dict)
    card_bal_m3_agg.columns = ['CARD_M3'+('_').join(column).upper() for column in card_bal_m3_agg.columns.ravel()]

    card_bal_agg = card_bal_agg.merge(card_bal_m3_agg, on='SK_ID_CURR', how='left')
    card_bal_agg = card_bal_agg.reset_index()

    return card_bal_agg

In [11]:
def get_dataset():
    app_train = pd.read_csv(os.path.join(default_dir,'application_train.csv'))
    app_test = pd.read_csv(os.path.join(default_dir,'application_test.csv'))
    apps = pd.concat([app_train, app_test])

    prev = pd.read_csv(os.path.join(default_dir,'previous_application.csv'))
    bureau = pd.read_csv(os.path.join(default_dir,'bureau.csv'))
    bureau_bal = pd.read_csv(os.path.join(default_dir,'bureau_balance.csv'))

    pos_bal, install, card_bal = get_balance_data()

    return apps, prev, bureau, bureau_bal, pos_bal, install, card_bal

apps, prev, bureau, bureau_bal, pos_bal, install, card_bal = get_dataset()

In [12]:
def get_apps_all_with_all_agg(apps, prev, bureau, bureau_bal, pos_bal, install, card_bal):
    """
    Description :
    1.Data preparation , aggregation
    2.produce the finalized result
    """
    apps_all =  get_apps_processed(apps)
    prev_agg = get_prev_agg(prev)
    bureau_agg = get_bureau_agg(bureau, bureau_bal)
    pos_bal_agg = get_pos_bal_agg(pos_bal)
    install_agg = get_install_agg(install)
    card_bal_agg = get_card_bal_agg(card_bal)
    print('prev_agg shape:', prev_agg.shape, 'bureau_agg shape:', bureau_agg.shape )
    print('pos_bal_agg shape:', pos_bal_agg.shape, 'install_agg shape:', install_agg.shape, 'card_bal_agg shape:', card_bal_agg.shape)
    print('apps_all before merge shape:', apps_all.shape)

    # Join with apps_all
    apps_all = apps_all.merge(prev_agg, on='SK_ID_CURR', how='left')
    apps_all = apps_all.merge(bureau_agg, on='SK_ID_CURR', how='left')
    apps_all = apps_all.merge(pos_bal_agg, on='SK_ID_CURR', how='left')
    apps_all = apps_all.merge(install_agg, on='SK_ID_CURR', how='left')
    apps_all = apps_all.merge(card_bal_agg, on='SK_ID_CURR', how='left')

    print('apps_all after merge with all shape:', apps_all.shape)

    return apps_all

In [13]:
def get_apps_all_with_prev_agg(apps, prev):
    apps_all =  get_apps_processed(apps)
    prev_agg = get_prev_agg(prev)
    print('prev_agg shape:', prev_agg.shape)
    print('apps_all before merge shape:', apps_all.shape)
    apps_all = apps_all.merge(prev_agg, on='SK_ID_CURR', how='left')
    print('apps_all after merge with prev_agg shape:', apps_all.shape)

    return apps_all

def get_apps_all_encoded(apps_all, return_mappings=False):
    """Replicate original joint-table factorization, optionally recording mappings.

    Factorization is applied on the concatenated train + test table, exactly as
    the original LightGBM code did. Integer codes are nominal categories,
    NOT meaningful ordered values; missing categories receive code -1.
    """
    category_mappings = {}
    object_columns = apps_all.dtypes[apps_all.dtypes == "object"].index.tolist()
    for column in object_columns:
        codes, categories = pd.factorize(apps_all[column])
        apps_all[column] = codes
        category_mappings[column] = {
            "missing_code": -1,
            "code_to_category": {str(index): str(value) for index, value in enumerate(categories)},
        }
    if return_mappings:
        return apps_all, category_mappings
    return apps_all


In [14]:
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

def get_apps_all_train_test(apps_all):
    apps_all_train = apps_all[~apps_all['TARGET'].isnull()]
    apps_all_test = apps_all[apps_all['TARGET'].isnull()]

    apps_all_test = apps_all_test.drop('TARGET', axis=1)

    return apps_all_train, apps_all_test


### 0C. Lock the **actual** LightGBM configuration and CV protocol

The original model is **not** trained on the MinMax-scaled / polynomial
exploratory table. It is trained on the numerical feature matrix from the
**eight-file aggregated LightGBM pipeline**, excluding `SK_ID_CURR` and `TARGET`.

Use stratified **5-fold CV**, shuffled with seed 42, producing exactly one
validation score per labeled applicant (OOF predictions). AUC from a training
fold is **not** an independent estimate; use its held-out fold score.

Changes relative to the original training helper are deliberately limited:
explicit `random_state=42`, modern `lightgbm.early_stopping` callback, validation-
only AUC as the early-stopping measure, and saved fold IDs / predictions.
The classifier hyperparameters match the supplied notebook; `subsample=0.708`
has **no row-bagging effect unless `subsample_freq > 0` is also set**. We record
that limitation rather than silently changing the model.


In [15]:
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss, log_loss
from lightgbm import LGBMClassifier, early_stopping, log_evaluation

# Same major tuning values as the notebook being audited.
LIGHTGBM_BASELINE_PARAMS = {
    "n_estimators": 2000,
    "learning_rate": 0.015,
    "max_depth": 13,
    "num_leaves": 60,
    "colsample_bytree": 0.613,
    "subsample": 0.708,
    "max_bin": 407,
    "reg_alpha": 10,
    "reg_lambda": 15,
    "min_child_weight": 6,
    "min_child_samples": 165,
    "n_jobs": 4,
    "random_state": AUDIT_SEED,
    "verbosity": -1,
    "metric": "auc",
}
CV_N_SPLITS = 5
EARLY_STOPPING_ROUNDS = 100

def train_with_cross_validation(data, target, application_ids,
                                artifacts_dir=AUDIT_OUTPUT_DIR,
                                model_params=None, n_splits=CV_N_SPLITS,
                                early_stopping_rounds=EARLY_STOPPING_ROUNDS,
                                save_fold_models=SAVE_FOLD_MODELS):
    """Train 5 LightGBM models and obtain label-held-out *out-of-fold* scores.

    The engineered feature table is fixed before CV, matching the original
    implementation; this does not fix possible transductive preprocessing.
    """
    if not data.columns.is_unique:
        raise ValueError("Repeated feature names are not supported.")
    if len(data) != len(target) or len(data) != len(application_ids):
        raise ValueError("Features, labels, and IDs must have equal row counts.")
    if not target.isna().eq(False).all():
        raise ValueError("Labeled population has missing targets.")
    if target.nunique() != 2:
        raise ValueError("ROC-AUC requires both target classes.")

    config = LIGHTGBM_BASELINE_PARAMS.copy()
    if model_params:
        config.update(model_params)
    artifacts_dir = Path(artifacts_dir)
    artifacts_dir.mkdir(parents=True, exist_ok=True)
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=AUDIT_SEED)

    oof_scores = np.full(len(target), np.nan, dtype=np.float64)
    oof_fold = np.zeros(len(target), dtype=np.int16)
    fold_records = []
    model_paths = []
    last_fold_model = None

    for fold, (train_idx, val_idx) in enumerate(cv.split(data, target), start=1):
        X_train, y_train = data.iloc[train_idx], target.iloc[train_idx]
        X_val, y_val = data.iloc[val_idx], target.iloc[val_idx]

        clf = LGBMClassifier(**config)
        clf.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            eval_metric="auc",
            callbacks=[
                early_stopping(stopping_rounds=early_stopping_rounds,
                               first_metric_only=True, verbose=False),
                log_evaluation(period=0),
            ],
        )

        # A fitted model defaults to best_iteration_ when predicting.
        fold_scores = clf.predict_proba(X_val)[:, 1]
        oof_scores[val_idx] = fold_scores
        oof_fold[val_idx] = fold

        row = {
            "fold": fold,
            "n_train": len(train_idx),
            "n_validation": len(val_idx),
            "validation_default_rate": float(y_val.mean()),
            "roc_auc": float(roc_auc_score(y_val, fold_scores)),
            "pr_auc_average_precision": float(average_precision_score(y_val, fold_scores)),
            "brier_score": float(brier_score_loss(y_val, fold_scores)),
            "best_iteration": int(clf.best_iteration_ or config["n_estimators"]),
        }
        fold_records.append(row)
        print(f"Fold {fold}/{n_splits}: AUC={row['roc_auc']:.5f} "
              f"AP={row['pr_auc_average_precision']:.5f} "
              f"best_iter={row['best_iteration']}")

        if save_fold_models:
            path = artifacts_dir / f"lightgbm_fold_{fold:02d}.txt"
            clf.booster_.save_model(str(path))
            model_paths.append(str(path))
        last_fold_model = clf

    if np.isnan(oof_scores).any() or not np.all(oof_fold > 0):
        raise RuntimeError("OOF predictions are incomplete.")
    if len(np.unique(np.asarray(application_ids))) != len(application_ids):
        raise ValueError("Duplicate applicant IDs would make OOF joins ambiguous.")

    oof = pd.DataFrame({
        "SK_ID_CURR": np.asarray(application_ids),
        "TARGET": np.asarray(target),
        "cv_fold": oof_fold,
        "oof_pred_default": oof_scores,
    })
    folds = pd.DataFrame(fold_records)
    aggregate = {
        "n_labeled": int(len(target)),
        "default_rate": float(target.mean()),
        "roc_auc_oof": float(roc_auc_score(target, oof_scores)),
        "pr_auc_oof_average_precision": float(average_precision_score(target, oof_scores)),
        "brier_score_oof": float(brier_score_loss(target, oof_scores)),
        "log_loss_oof": float(log_loss(target, oof_scores, labels=[0, 1])),
        "fold_auc_mean": float(folds["roc_auc"].mean()),
        "fold_auc_std_sample": float(folds["roc_auc"].std(ddof=1)),
        "cv_splits": int(n_splits),
        "seed": int(AUDIT_SEED),
        "early_stopping_rounds": int(early_stopping_rounds),
        "lightgbm_params": config,
        "model_paths": model_paths,
        "evaluation_note": "All pooled scores use OOF predictions, not training scores.",
    }
    oof.to_csv(artifacts_dir / "oof_predictions.csv", index=False)
    folds.to_csv(artifacts_dir / "fold_metrics.csv", index=False)
    (artifacts_dir / "baseline_metrics.json").write_text(
        json.dumps(aggregate, indent=2), encoding="utf-8"
    )
    print(f"OOF ROC-AUC={aggregate['roc_auc_oof']:.5f}; "
          f"OOF PR-AUC={aggregate['pr_auc_oof_average_precision']:.5f}")
    return {
        "last_fold_model": last_fold_model,  # NOT fitted to the entire training set
        "oof_predictions": oof,
        "fold_metrics": folds,
        "baseline_metrics": aggregate,
        "model_paths": model_paths,
    }


### 0D. Construct and inspect the final LightGBM feature matrix

The original `get_dataset()` re-reads raw CSVs and the aggregate functions join
previous applications, bureau records, bureau balances, POS/CASH balances,
installment payments, and credit-card balances by applicant ID. That feature
matrix—not the earlier imputed/scaled Random Forest table—is the subject of
the audit. **Do not remove proxy variables or change preprocessing in Phase 0**:
we first measure the original implementation as-is, then test modifications.


In [16]:
# Reproduce exactly the final LightGBM feature construction sequence.
apps_all = get_apps_all_with_all_agg(apps, prev, bureau, bureau_bal,
                                     pos_bal, install, card_bal)
apps_all, category_mappings = get_apps_all_encoded(apps_all, return_mappings=True)
apps_all_train, apps_all_test = get_apps_all_train_test(apps_all)

ftr_app = apps_all_train.drop(columns=["SK_ID_CURR", "TARGET"])
target_app = apps_all_train["TARGET"].astype(int)
train_application_ids = apps_all_train["SK_ID_CURR"].copy()

assert len(apps_all_train) == len(app_train), "Labeled applicant count changed during joins."
assert len(apps_all_test) == len(app_test), "Test applicant count changed during joins."
assert train_application_ids.is_unique and apps_all_test["SK_ID_CURR"].is_unique
assert list(ftr_app.columns) == list(apps_all_test.drop(columns=["SK_ID_CURR"]).columns), (
    "Training and competition-test feature order differs."
)
assert not ftr_app.columns.duplicated().any(), "Duplicate feature names."
assert not ftr_app.select_dtypes(exclude="number").columns.size, "Unencoded features remain."
assert "TARGET" not in ftr_app and "SK_ID_CURR" not in ftr_app
assert target_app.isin([0, 1]).all()
assert set(train_application_ids) == set(audit_reference_raw["SK_ID_CURR"])

# Audit potentially problematic inputs without silently changing the baseline.
missing_frac = ftr_app.isna().mean().sort_values(ascending=False)
infinite_counts = {
    col: int(np.isinf(ftr_app[col].to_numpy()).sum())
    for col in ftr_app.columns if pd.api.types.is_float_dtype(ftr_app[col])
}
nonzero_inf = {k: v for k, v in infinite_counts.items() if v > 0}
feature_summary = {
    "train_rows": int(len(ftr_app)),
    "competition_test_rows": int(len(apps_all_test)),
    "n_features": int(ftr_app.shape[1]),
    "feature_names_in_order": ftr_app.columns.to_list(),
    "n_features_with_missing_values": int((missing_frac > 0).sum()),
    "n_features_with_infinite_values": len(nonzero_inf),
    "infinite_value_counts": nonzero_inf,
    "n_factorized_object_columns": len(category_mappings),
    "excluded_from_model": ["SK_ID_CURR", "TARGET"],
    "feature_source": "get_apps_all_with_all_agg + get_apps_all_encoded, eight raw CSVs",
    "notes": [
        "Joint train+competition-test preprocessing (factorize and some derived means) is retained.",
        "Integer-coded nominal categories are treated as numeric by LightGBM.",
        "Potential post-application information in auxiliary tables has not been ruled out.",
        "Original engineered ratios and missing-value behavior remain unchanged.",
        "DAYS_EMPLOYED=365243 remains in the actual LightGBM path even if the EDA path cleans it.",
        "The earlier exploratory imputation, scaling and Random Forest are NOT LightGBM inputs.",
    ],
}
(AUDIT_OUTPUT_DIR / "feature_schema.json").write_text(
    json.dumps(feature_summary, indent=2), encoding="utf-8"
)
(AUDIT_OUTPUT_DIR / "category_codebook.json").write_text(
    json.dumps(category_mappings, indent=2), encoding="utf-8"
)
missing_frac.rename("missing_fraction").to_csv(AUDIT_OUTPUT_DIR / "feature_missingness.csv")
print(f"LightGBM feature table: {len(ftr_app):,} rows × {ftr_app.shape[1]:,} features")
print(f"Factorized original object columns: {len(category_mappings)}")
print(f"Features with +/-infinity: {len(nonzero_inf)}")
display(missing_frac.head(12).rename("missing_fraction").to_frame())
if nonzero_inf:
    print("Flagged infinity counts (not altered):", sorted(nonzero_inf.items(),
                                                           key=lambda item: -item[1])[:12])


prev_agg shape: (338857, 80) bureau_agg shape: (305811, 103)
pos_bal_agg shape: (337252, 45) install_agg shape: (339587, 59) card_bal_agg shape: (103558, 70)
apps_all before merge shape: (356255, 135)
apps_all after merge with all shape: (356255, 488)
LightGBM feature table: 307,511 rows × 486 features
Factorized original object columns: 16
Features with +/-infinity: 27


,missing_fraction
BUREAU_ACT_BUREAU_ENDDATE_FACT_DIFF_MIN,0.995200
BUREAU_ACT_BUREAU_ENDDATE_FACT_DIFF_MAX,0.995200
BUREAU_ACT_BUREAU_ENDDATE_FACT_DIFF_MEAN,0.995200
BUREAU_ACT_DAYS_ENDDATE_FACT_MEAN,0.995171
BUREAU_ACT_DAYS_ENDDATE_FACT_MIN,0.995171
BUREAU_ACT_BUREAU_CREDIT_FACT_DIFF_MIN,0.995171
BUREAU_ACT_DAYS_ENDDATE_FACT_MAX,0.995171
BUREAU_ACT_BUREAU_CREDIT_FACT_DIFF_MAX,0.995171
BUREAU_ACT_BUREAU_CREDIT_FACT_DIFF_MEAN,0.995171
PREV_D365_PREV_DAYS_LAST_DUE_DIFF_MEAN,0.860337


Flagged infinity counts (not altered): [('PREV_PREV_INTERESTS_RATE_MEAN', 96838), ('PREV_PREV_CREDIT_APPL_RATIO_MEAN', 42894), ('PREV_PREV_CREDIT_APPL_RATIO_MAX', 42894), ('PREV_D365_PREV_INTERESTS_RATE_MEAN', 37658), ('PREV_D365_PREV_INTERESTS_RATE_MAX', 8696), ('BUREAU_BUREAU_CREDIT_DEBT_RATIO_MEAN', 4142), ('BUREAU_ACT_BUREAU_CREDIT_DEBT_RATIO_MEAN', 4048), ('CARD_DRAWING_LIMIT_RATIO_MAX', 3712), ('BUREAU_BUREAU_CREDIT_DEBT_RATIO_MIN', 2560), ('BUREAU_ACT_BUREAU_CREDIT_DEBT_RATIO_MIN', 2545), ('PREV_PREV_INTERESTS_RATE_MAX', 2075), ('BUREAU_ACT_BUREAU_CREDIT_DEBT_RATIO_MAX', 1883)]


### 0E. Evaluate and freeze the OOF baseline

The CV procedure trains one model per fold with the notebook's LightGBM
hyperparameters. Predictions for **each applicant come only from a model that
did not train on that applicant**. Save the fold assignments and model snapshots
to support exactly matched stability analyses in subsequent phases.

The metric reported for an applicant is **default-risk score** (`TARGET=1`):
higher means *more* predicted risk of defaulting, not more creditworthiness. Since there is
no specified real lender decision policy, we **do not assume that 0.5 is a
loan rejection threshold**.

In [17]:
phase0_result = train_with_cross_validation(
    ftr_app, target_app, train_application_ids,
    artifacts_dir=AUDIT_OUTPUT_DIR,
)
oof_predictions = phase0_result["oof_predictions"]
fold_metrics = phase0_result["fold_metrics"]
baseline_metrics = phase0_result["baseline_metrics"]

# Attach *raw* attributes by ID, never rely on row order after merges.
audit_oof = oof_predictions.merge(
    audit_reference_raw, on="SK_ID_CURR", how="left", validate="one_to_one"
)
assert len(audit_oof) == len(oof_predictions)
assert audit_oof["SK_ID_CURR"].is_unique
audit_oof.to_csv(AUDIT_OUTPUT_DIR / "audit_oof_with_raw_attributes.csv", index=False)

# Convenience aliases for future SHAP/robustness notebook cells.
# LAST FOLD MODEL is *not* a full-data fit, and should NOT be called final_model.
last_cv_fold_model = phase0_result["last_fold_model"]
print("Fold-level performance:")
display(fold_metrics)
print("Pooled, held-out baseline metrics:")
display(pd.DataFrame([{
    name: value for name, value in baseline_metrics.items()
    if isinstance(value, (int, float))
}]).T.rename(columns={0: "value"}))
print("Artifacts written to:", AUDIT_OUTPUT_DIR.resolve())


Fold 1/5: AUC=0.79295 AP=0.28958 best_iter=1687
Fold 2/5: AUC=0.80277 AP=0.30315 best_iter=1591
Fold 3/5: AUC=0.79362 AP=0.28246 best_iter=1715
Fold 4/5: AUC=0.79749 AP=0.29815 best_iter=1401
Fold 5/5: AUC=0.79287 AP=0.28642 best_iter=1752
OOF ROC-AUC=0.79590; OOF PR-AUC=0.29167
Fold-level performance:


,fold,n_train,n_validation,validation_default_rate,roc_auc,pr_auc_average_precision,brier_score,best_iteration
0,1,246008,61503,0.080728,0.792947,0.289583,0.065431,1687
1,2,246009,61502,0.080729,0.802768,0.303153,0.064784,1591
2,3,246009,61502,0.080729,0.793616,0.282462,0.065690,1715
3,4,246009,61502,0.080729,0.797488,0.298150,0.065020,1401
4,5,246009,61502,0.080729,0.792866,0.286417,0.065653,1752


Pooled, held-out baseline metrics:


,value
n_labeled,307511.000000
default_rate,0.080729
roc_auc_oof,0.795898
pr_auc_oof_average_precision,0.291665
brier_score_oof,0.065316
log_loss_oof,0.234126
fold_auc_mean,0.795937
fold_auc_std_sample,0.004267
cv_splits,5.000000
seed,42.000000


Artifacts written to: /content/audit_artifacts/phase0


### 0F. Phase 0 acceptance checks and audit handoff

Checks below run **after** 0E completes. Once they pass, use
`audit_artifacts/phase0/` as the frozen reference for subsequent phases:
- **SHAP/interpretability:** `feature_schema.json`, `category_codebook.json`,
  and saved LightGBM fold models. For exact per-applicant attributions, use the
  fold model identified by `cv_fold`, not a model trained on that applicant.
- **Stability/robustness:** the frozen `ftr_app`, feature names in order, and
  each applicant's `oof_pred_default` as their original baseline score.
- **Important cases/subpopulations:** `audit_oof_with_raw_attributes.csv`,
  with unmodified demographic/context variables preserved for comparisons.

**Known validity questions to test in later phases:** original preprocessing
factors categories using both labeled and competition-test rows (transductive);
missing categories are encoded as `-1` and interpreted as numeric; the main
LightGBM path retains the `DAYS_EMPLOYED=365243` sentinel; engineered ratios
may be infinite; some auxiliary credit-history fields may not be available at
the intended decision time; and the competition test labels are unavailable.
None is silently repaired here, because Phase 0 is the **reference model**.

A successful Phase 0 run creates `phase0_run_complete.json` only after the
integrity checks below pass. Source files alone do not establish a measured
baseline; all metrics require the original eight competition datasets.


In [18]:
assert len(oof_predictions) == len(target_app)
assert oof_predictions["SK_ID_CURR"].is_unique
assert oof_predictions["cv_fold"].nunique() == CV_N_SPLITS
assert int(fold_metrics["n_validation"].sum()) == len(target_app)
assert oof_predictions["oof_pred_default"].between(0, 1).all()
assert oof_predictions["oof_pred_default"].notna().all()
assert np.isfinite(baseline_metrics["roc_auc_oof"])
assert np.isfinite(baseline_metrics["pr_auc_oof_average_precision"])
assert len(audit_oof) == len(oof_predictions)

expected_files = [
    "environment.json", "raw_file_inventory.csv", "raw_population_summary.json",
    "feature_schema.json", "category_codebook.json", "feature_missingness.csv",
    "oof_predictions.csv", "audit_oof_with_raw_attributes.csv",
    "fold_metrics.csv", "baseline_metrics.json",
]
if SAVE_FOLD_MODELS:
    expected_files.extend(f"lightgbm_fold_{fold:02d}.txt"
                          for fold in range(1, CV_N_SPLITS + 1))
missing_exports = [name for name in expected_files
                   if not (AUDIT_OUTPUT_DIR / name).is_file()]
assert not missing_exports, f"Missing Phase 0 exports: {missing_exports}"

completion_record = {
    "completed_utc": pd.Timestamp.now(tz="UTC").isoformat(),
    "status": "Passed Phase 0 runtime checks",
    "data_dir": str(default_dir),
    "artifact_dir": str(AUDIT_OUTPUT_DIR.resolve()),
    "n_labeled": int(len(target_app)),
    "n_features": int(ftr_app.shape[1]),
    "cv_splits": int(CV_N_SPLITS),
    "primary_metric": "OOF ROC-AUC",
    "secondary_metrics": ["OOF average precision", "OOF Brier score",
                          "OOF log loss", "fold ROC-AUC mean/std"],
    "saved_artifacts": expected_files,
}
(AUDIT_OUTPUT_DIR / "phase0_run_complete.json").write_text(
    json.dumps(completion_record, indent=2), encoding="utf-8"
)
print("Phase 0 passed: baseline and audit artifacts saved.")
display(pd.DataFrame({"artifact": expected_files,
                      "verified": [True] * len(expected_files)}))


Phase 0 passed: baseline and audit artifacts saved.


,artifact,verified
0,environment.json,True
1,raw_file_inventory.csv,True
2,raw_population_summary.json,True
3,feature_schema.json,True
4,category_codebook.json,True
5,feature_missingness.csv,True
6,oof_predictions.csv,True
7,audit_oof_with_raw_attributes.csv,True
8,fold_metrics.csv,True
9,baseline_metrics.json,True


I used ChatGPT to help write code to test the original model's reproducibility.